In [16]:
import os 
import numpy as np  
import matplotlib.pyplot as plt
import pandas as pd 
from PIL import Image   
import torch 
from torchvision import models, transforms


from sklearn.model_selection import train_test_split    
from sklearn.svm import OneClassSVM
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report, 
    precision_score, 
    recall_score,
    f1_score
)

In [15]:
einsr = "../einstein_rings"
image_folder = "../data_labeled/Einstein_Rings"
csv_file = "labels.csv"
img_size = (64, 64)

image_size = 224

valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

image_paths = []
source_labels = []   # 1 = einstein, 0 = unknown

for fname in os.listdir(einsr):
    if fname.lower().endswith(valid_exts):
        image_paths.append(os.path.join(einsr, fname))
        source_labels.append(1)

for fname in os.listdir(image_folder):
    if fname.lower().endswith(valid_exts):
        image_paths.append(os.path.join(image_folder, fname))
        source_labels.append(0)

print("Total images collected:", len(image_paths))
print("Einstein rings from Amal:", sum(source_labels))
print("Einstein rings from raw dataset:", len(source_labels) - sum(source_labels))

Total images collected: 195
Einstein rings from Amal: 37
Einstein rings from raw dataset: 158


## Feature Extractor with ResNet

In [19]:
train_folder = "../einstein_rings"
review_folder = "../data_labeled/Einstein_Rings"

# Feature Extractor 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT )
model = torch.nn.Sequential(*list(model.children())[:-1]) # Remove the final classification layer   
model.eval().to(device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

def extract_features(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        features = model(image).squeeze().cpu().numpy()
    
    return features

# Load train features 
train_paths = [
    os.path.join(train_folder, f)
    for f in os.listdir(train_folder)
    if f.lower().endswith(valid_exts)
]
X_train = np.array([extract_features(p) for p in train_paths])
y_train = np.ones(len(X_train))  # All are Einstein rings
print("Extracted features for training:", X_train.shape)
print("Training labels shape:", y_train.shape)




Extracted features for training: (37, 512)
Training labels shape: (37,)


## 1. Training with one-class SVM

In [24]:
# Train one-class SVM
ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.1)
ocsvm.fit(X_train)
# Load review features
review_paths = [
    os.path.join(review_folder, f)
    for f in os.listdir(review_folder)
    if f.lower().endswith(valid_exts)
]
results = []
# Predict and score review images
# pred = 1 -> looks like Einstein ring, pred = -1 -> unusual image, need to review manually, 
# score = how strongly it looks like an Einstein ring (higher is more similar)
for p in review_paths:
    feat = extract_features(p).reshape(1, -1)
    pred = ocsvm.predict(feat)[0]          # 1 = inlier, -1 = outlier
    score = ocsvm.decision_function(feat)[0]

    results.append({
        "filename": os.path.basename(p),
        "pred": pred,
        "score": score,
        "status": "correct_like_einstein" if pred == 1 else "outlier_check_manually"
    })

df_results = pd.DataFrame(results)
df_results = df_results.sort_values("score")

total = len(df_results)

summary_df = df_results["pred"].value_counts().rename({
    1: "Inlier (Einstein-like)",
    -1: "Outlier (Check)"
}).reset_index()

summary_df.columns = ["Category", "Count"]
summary_df["Percentage (%)"] = (summary_df["Count"] / total) * 100

print(summary_df)

print(df_results.head(20))


                 Category  Count  Percentage (%)
0  Inlier (Einstein-like)    106       67.088608
1         Outlier (Check)     52       32.911392
                                              filename  pred     score  \
135  102159483_2659872799666371332_102159483_265987...    -1 -0.487194   
54   102021987_NEG632575884473969454_102021987_NEG6...    -1 -0.276698   
85   102044824_NEG525230483273308137_102044824_NEG5...    -1 -0.242350   
134  102159196_2750719383660442039_102159196_275071...    -1 -0.234583   
89   102045464_NEG514418387268262939_102045464_NEG5...    -1 -0.231222   
93   102046111_NEG524535231263409554_102046111_NEG5...    -1 -0.220578   
124  102158892_2704470582654415692_102158892_270447...    -1 -0.213220   
77   102042919_NEG550105155291238806_102042919_NEG5...    -1 -0.198978   
25   102021014_NEG614981615482506340_102021014_NEG6...    -1 -0.186637   
103  102157956_2700546205640397835_102157956_270054...    -1 -0.178188   
95   102046114_NEG537352034266671160_10

## 2. Training with CNN Autoencoder 

In [25]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class ImageDataset(Dataset):
    def __init__(self, folder):
        self.paths = [
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.lower().endswith((".jpg",".png",".jpeg",".bmp",".webp"))
        ]
        self.transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.Resize((64,64)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("L")
        return self.transform(img)

In [26]:
import torch.nn as nn

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 64 -> 32

            nn.Conv2d(16,32,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 32 -> 16

            nn.Conv2d(32,64,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2)   # 16 -> 8
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64,32,2,stride=2), nn.ReLU(),
            nn.ConvTranspose2d(32,16,2,stride=2), nn.ReLU(),
            nn.ConvTranspose2d(16,1,2,stride=2),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_folder = "../einstein_rings"

dataset = ImageDataset(train_folder)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = Autoencoder().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 20

for epoch in range(epochs):
    total_loss = 0

    for imgs in loader:
        imgs = imgs.to(device)

        outputs = model(imgs)
        loss = criterion(outputs, imgs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 0.8359
Epoch 2, Loss: 0.7865
Epoch 3, Loss: 0.7028
Epoch 4, Loss: 0.5196
Epoch 5, Loss: 0.3276
Epoch 6, Loss: 0.2729
Epoch 7, Loss: 0.2805
Epoch 8, Loss: 0.2359
Epoch 9, Loss: 0.1821
Epoch 10, Loss: 0.1607
Epoch 11, Loss: 0.1202
Epoch 12, Loss: 0.0912
Epoch 13, Loss: 0.0721
Epoch 14, Loss: 0.0613
Epoch 15, Loss: 0.0564
Epoch 16, Loss: 0.0550
Epoch 17, Loss: 0.0540
Epoch 18, Loss: 0.0522
Epoch 19, Loss: 0.0508
Epoch 20, Loss: 0.0491


In [28]:
import pandas as pd

review_folder = "../data_labeled/Einstein_Rings"

review_dataset = ImageDataset(review_folder)
review_loader = DataLoader(review_dataset, batch_size=1)

results = []

model.eval()

with torch.no_grad():
    for i, img in enumerate(review_loader):
        img = img.to(device)

        output = model(img)
        loss = torch.mean((img - output)**2).item()

        results.append({
            "filename": review_dataset.paths[i],
            "reconstruction_error": loss
        })

df_results = pd.DataFrame(results)
df_results = df_results.sort_values("reconstruction_error", ascending=False)

print(df_results.head(20))

                                              filename  reconstruction_error
95   ../data_labeled/Einstein_Rings\102046114_NEG53...              0.022132
100  ../data_labeled/Einstein_Rings\102157953_26709...              0.021187
135  ../data_labeled/Einstein_Rings\102159483_26598...              0.019348
18   ../data_labeled/Einstein_Rings\102020532_NEG59...              0.015639
105  ../data_labeled/Einstein_Rings\102157958_27271...              0.013774
78   ../data_labeled/Einstein_Rings\102042919_NEG55...              0.013218
88   ../data_labeled/Einstein_Rings\102045463_NEG51...              0.013201
140  ../data_labeled/Einstein_Rings\102159776_27070...              0.013094
142  ../data_labeled/Einstein_Rings\102159778_27334...              0.013003
126  ../data_labeled/Einstein_Rings\102158894_27306...              0.012812
54   ../data_labeled/Einstein_Rings\102021987_NEG63...              0.012697
118  ../data_labeled/Einstein_Rings\102158586_27079...              0.012565

In [31]:
threshold = df_results["reconstruction_error"].quantile(0.9)

df_results["pred"] = df_results["reconstruction_error"].apply(
    lambda x: "outlier" if x > threshold else "inlier"
)
summary = df_results["pred"].value_counts().reset_index()   
summary.columns = ["Category", "Count"]
summary["Percentage (%)"] = (summary["Count"] / len(df_results)) * 100
print(summary)


  Category  Count  Percentage (%)
0   inlier    142       89.873418
1  outlier     16       10.126582
